In [1]:
import pandas as pd
import json

# Đọc file JSONL
df = pd.read_json("shopee_reviews_dataset.jsonl", lines=True)

# Xem thông tin cơ bản
print("Số lượng bản ghi:", len(df))
print("\nCác cột hiện có:")
print(df.columns.tolist())
print("\n5 dòng đầu tiên:")
print(df.head())
print("\nThông tin dữ liệu:")
print(df.info())

Số lượng bản ghi: 9599

Các cột hiện có:
['id', 'review', 'rating', 'label']

5 dòng đầu tiên:
            id                                             review  rating  \
0  74263765409  Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy...       3   
1  11104151002  Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...       5   
2  15888299382  Chất lượng sản phẩm:ok Đúng với mô tả:đúng  Lầ...       5   
3  81030214453  Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...       5   
4  88484377297  Hương vị:Mix vị, Tím  Mình mua 64 gói (32 gói ...       1   

      label  
0  negative  
1  positive  
2  positive  
3  positive  
4  negative  

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9599 entries, 0 to 9598
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      9599 non-null   int64 
 1   review  9599 non-null   object
 2   rating  9599 non-null   int64 
 3   label   9599 non-null   object
dtypes: int64(2)

In [2]:
# Kiểm tra các cột
print(df.columns)

# Phân bố nhãn cảm xúc (nếu có cột label)
if "label" in df.columns:
    print("\nPhân bố nhãn:")
    print(df["label"].value_counts())
    print(df["label"].value_counts(normalize=True) * 100)

# Phân bố rating (nếu có)
if "rating" in df.columns:
    print("\nPhân bố rating:")
    print(df["rating"].value_counts().sort_index())

# Kiểm tra giá trị thiếu
print("\nSố lượng giá trị thiếu:")
print(df.isnull().sum())

# Độ dài trung bình của đánh giá
df["review_length"] = df["review"].astype(str).apply(len)
print("\nThống kê độ dài đánh giá:")
print(df["review_length"].describe())

Index(['id', 'review', 'rating', 'label'], dtype='object')

Phân bố nhãn:
label
negative    5965
positive    3634
Name: count, dtype: int64
label
negative    62.14189
positive    37.85811
Name: proportion, dtype: float64

Phân bố rating:
rating
1    2972
2    1230
3    1782
4      96
5    3519
Name: count, dtype: int64

Số lượng giá trị thiếu:
id        0
review    0
rating    0
label     0
dtype: int64

Thống kê độ dài đánh giá:
count    9599.000000
mean      133.919575
std       109.318574
min        20.000000
25%        71.000000
50%       106.000000
75%       159.000000
max      2021.000000
Name: review_length, dtype: float64


In [3]:
def create_3class_label(row):
    if row["label"] == "positive":
        return "positive"
    elif row["label"] == "negative":
        return "negative"
    # Trường hợp đặc biệt: nếu rating = 3 thì ưu tiên Neutral
    if row["rating"] == 3:
        return "neutral"
    return row["label"]

# Tạo cột nhãn 3 lớp
df["label_3class"] = df.apply(create_3class_label, axis=1)

# Kiểm tra phân bố mới
print(df["label_3class"].value_counts())
print(df["label_3class"].value_counts(normalize=True).round(4) * 100)

label_3class
negative    5965
positive    3634
Name: count, dtype: int64
label_3class
negative    62.14
positive    37.86
Name: proportion, dtype: float64


In [4]:
# Loại bỏ đánh giá quá ngắn
df_clean = df[df["review_length"] >= 10].copy()

# Loại bỏ trùng lặp (nếu có)
df_clean = df_clean.drop_duplicates(subset=["review"])

print("Số bản ghi sau khi làm sạch:", len(df_clean))

Số bản ghi sau khi làm sạch: 9599


In [6]:
def create_3class_label(row):
    if row["rating"] == 3:
        return "neutral"
    elif row["label"] == "positive":
        return "positive"
    else:
        return "negative"

df_clean["label_3class"] = df_clean.apply(create_3class_label, axis=1)

print("Phân bố nhãn 3 lớp:")
print(df_clean["label_3class"].value_counts())
print("\nTỷ lệ phần trăm:")
print(df_clean["label_3class"].value_counts(normalize=True).round(4) * 100)

Phân bố nhãn 3 lớp:
label_3class
negative    4274
positive    3543
neutral     1782
Name: count, dtype: int64

Tỷ lệ phần trăm:
label_3class
negative    44.53
positive    36.91
neutral     18.56
Name: proportion, dtype: float64


In [7]:
# Loại bỏ đánh giá quá ngắn
df_clean = df_clean[df_clean["review"].astype(str).str.len() >= 10].copy()

# Loại bỏ trùng lặp nội dung
df_clean = df_clean.drop_duplicates(subset=["review"]).reset_index(drop=True)

print("Số bản ghi sau khi làm sạch cuối cùng:", len(df_clean))
print("\nPhân bố nhãn sau làm sạch:")
print(df_clean["label_3class"].value_counts())

Số bản ghi sau khi làm sạch cuối cùng: 9599

Phân bố nhãn sau làm sạch:
label_3class
negative    4274
positive    3543
neutral     1782
Name: count, dtype: int64


In [8]:
import re

def preprocess_text(text):
    text = str(text).lower()                          # Chữ thường
    text = re.sub(r'\s+', ' ', text)                  # Chuẩn hóa khoảng trắng
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', ' ', text)  # Giữ chữ cái tiếng Việt + số
    text = text.strip()
    return text

df_clean["review_cleaned"] = df_clean["review"].apply(preprocess_text)

# Kiểm tra kết quả
print(df_clean[["review", "review_cleaned"]].head(5))

                                              review  \
0  Hương vị:thom  Chắc do bên giao hàng bị vỡ mấy...   
1  Hương thơm:nhẹ nhàng Lợi ích:phục hồi cấp ẩm M...   
2  Chất lượng sản phẩm:ok Đúng với mô tả:đúng  Lầ...   
3  Độ tuổi sử dụng:em bes Chất lượng sản phẩm:tot...   
4  Hương vị:Mix vị, Tím  Mình mua 64 gói (32 gói ...   

                                      review_cleaned  
0  hương vị thom chắc do bên giao hàng bị vỡ mấy gói  
1  hương thơm nhẹ nhàng lợi ích phục hồi cấp ẩm m...  
2  chất lượng sản phẩm ok đúng với mô tả đúng lần...  
3  độ tuổi sử dụng em bes chất lượng sản phẩm tot...  
4  hương vị mix vị  tím mình mua 64 gói  32 gói m...  


In [9]:
# Chạy 1 lần duy nhất
!pip install underthesea -q



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\viloc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [10]:
from underthesea import word_tokenize

df_clean["review_tokenized"] = df_clean["review_cleaned"].apply(lambda x: word_tokenize(x, format="text"))

In [11]:
from sklearn.model_selection import train_test_split

# Chia tập
train_df, temp_df = train_test_split(
    df_clean, 
    test_size=0.3, 
    random_state=42, 
    stratify=df_clean["label_3class"]
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df["label_3class"]
)

print("Kích thước các tập:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

print("\nPhân bố nhãn tập Train:")
print(train_df["label_3class"].value_counts(normalize=True).round(3) * 100)

Kích thước các tập:
Train: 6719
Validation: 1440
Test: 1440

Phân bố nhãn tập Train:
label_3class
negative    44.5
positive    36.9
neutral     18.6
Name: proportion, dtype: float64


In [12]:
# Lưu lại để sử dụng sau
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

df_clean.to_csv("shopee_reviews_cleaned_3class.csv", index=False)

print("Đã lưu các file dữ liệu thành công!")

Đã lưu các file dữ liệu thành công!


In [13]:
!pip install transformers datasets accelerate -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

ERROR: Could not install packages due to an OSError: [WinError 206] The filename or extension is too long: 'C:\\Users\\viloc\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\torch-2.14.1.dist-info\\licenses\\third_party\\flash-attention\\third_party\\aiter\\3rdparty\\composable_kernel\\docs'


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\viloc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\viloc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
